# Experiment 10 (corrected pipeline: training data shuffled) (grouped 5-fold CV): ResNet50, improved recipe

Five-fold cross-validation with the **same improved recipe** as Experiments 3-4. Folds: GROUPED folds: all slices of one near-duplicate group (scan) stay in the same fold, so the test fold contains only unseen scans.
Every image is tested exactly once (out-of-fold). For fold k the test set is fold k, the validation set is one other fold (used only for early stopping and checkpoint selection), and the rest is training (about 60/20/20).

Why: Experiment 5a showed that the original split is heavily leaky (1-nearest-neighbour on thumbnails scores 99.9%). Running the identical protocol with grouped and with random folds measures how much accuracy is inflated by near-duplicate slices.

**Kaggle setup:** add the same Alzheimer dataset as input, Accelerator **GPU T4 x2 or P100**, Internet ON, *Save Version -> Save & Run All (Commit)*. Runtime: about 28 min per fold, about 2.5 hours in total.
The cell output shows the near-duplicate threshold (for the rule-based grouped mode expect tau = 0.8 and 213 groups; for a fixed-tau run it prints the tau given: None). Fixed-tau runs (7e/7f) are the leakage dose-response experiment: the higher the tau, the finer the groups and the more near-duplicate leakage remains.

In [ ]:
# 1. Environment and configuration
import os, json, time, random, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)

DRY_RUN = os.environ.get("DRY_RUN") == "1"          # only used by the author for a CPU smoke test
DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")
WEIGHTS = None if DRY_RUN else "imagenet"
IMG_SIZE = (64, 64) if DRY_RUN else (224, 224)
BATCH_SIZE = 8 if DRY_RUN else 32
P1_EPOCHS = 1 if DRY_RUN else 8        # Phase 1: backbone frozen, head only
P2_EPOCHS = 1 if DRY_RUN else 30       # Phase 2: fine-tuning with cosine decay
SEEDS = [42] if DRY_RUN else [42, 123, 456, 789, 2024]
ARCHS = ["resnet50"]
VARIANTS = ["full"]
SPLIT_MODE = "grouped"
TAG = "exp11b_resnet50_gcv"
MAX_GROUP_FRAC = 0.05
N_FOLDS = 5
FOLDS = [0] if DRY_RUN else list(range(N_FOLDS))
SEED = 2042
TAU_FIXED = None   # v3: ConvNeXt-Tiny (~100 min/run) and EfficientNetV2-S were dropped from the 5-seed run to save GPU quota;
                        # add "efficientnetv2_s" or "convnext_tiny" here to test them
LABEL_SMOOTHING = 0.1
SPLIT_SEED = 42

if not DRY_RUN:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")   # faster on T4/P100, softmax stays float32

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Seeds:", SEEDS, "| Architectures:", ARCHS)


In [ ]:
# 2. Locate dataset and build the file index (same logic as Experiments 1-2)
CLASSES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
label_to_idx = {c: i for i, c in enumerate(CLASSES)}

def normalize(name):
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")

canonical_by_norm = {normalize(c): c for c in CLASSES}
class_dirs = {}
for root, dirs, files in os.walk(DATA_ROOT):
    norm_root_name = normalize(os.path.basename(root))
    if norm_root_name in canonical_by_norm and any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
        class_dirs.setdefault(canonical_by_norm[norm_root_name], []).append(root)
assert all(c in class_dirs for c in CLASSES), "Could not find all 4 class folders - check the dataset input."

records = []
for cls, dirs_for_cls in class_dirs.items():
    for cls_dir in dirs_for_cls:
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                records.append({"filepath": os.path.join(cls_dir, fname), "label": cls})
df = pd.DataFrame(records).drop_duplicates(subset="filepath").reset_index(drop=True)
print("Total images:", len(df))
print(df["label"].value_counts().to_string())


In [ ]:
# --- Near-duplicate-aware (grouped) split: deterministic, CPU-only, no patient IDs needed ---
from PIL import Image

def pixel_features(paths, side=64):
    """64x64 grayscale thumbnails; per-image mean removed, then the dataset-mean image removed (so the shared head/brain template
    does not dominate), then L2-normalised: a dot product is a correlation, 1.0 = identical up to brightness/contrast."""
    X = np.empty((len(paths), side * side), dtype=np.float64)
    for i, p in enumerate(paths):
        im = Image.open(p).convert("L").resize((side, side), Image.BOX)
        X[i] = np.asarray(im, dtype=np.float64).ravel()
    X -= X.mean(axis=1, keepdims=True)
    X -= X.mean(axis=0, keepdims=True)
    X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    return X

def threshold_groups(X, max_frac=0.05, ladder=(0.999, 0.995, 0.99, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50)):
    """Near-duplicate groups = connected components of the graph 'correlation > tau'. tau is chosen by a rule fixed in advance:
    the LOWEST tau on the ladder whose largest component holds at most max_frac of all images (lower tau = stricter separation)."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components
    n = len(X); S = np.round(X @ X.T, 6); np.fill_diagonal(S, -1.0)
    table = []; chosen = None
    for tau in ladder:
        ncomp, lab = connected_components(csr_matrix(S > tau), directed=False)
        sizes = np.bincount(lab)
        table.append({"tau": tau, "n_groups": int(ncomp), "largest_group": int(sizes.max()), "singletons": int((sizes == 1).sum())})
        if sizes.max() <= max_frac * n: chosen = (tau, lab)
    if chosen is None: chosen = (ladder[0], connected_components(csr_matrix(S > ladder[0]), directed=False)[1])
    return chosen[1], chosen[0], table

def groups_at_tau(X, tau):
    """Connected components of 'correlation > tau' for a FIXED tau (used for the leakage dose-response experiment)."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components
    S = np.round(X @ X.T, 6); np.fill_diagonal(S, -1.0)
    return connected_components(csr_matrix(S > tau), directed=False)[1]

def assign_groups(groups, labels, n_cls=4, fractions=(0.70, 0.15, 0.15)):
    """Greedy, deterministic assignment of whole groups to train/val/test that keeps per-class proportions close to 70/15/15.
    Groups containing rare classes are placed first."""
    tot = np.bincount(labels, minlength=n_cls).astype(float)
    target = np.array([tot * f for f in fractions])
    cur = np.zeros_like(target)
    gids = np.unique(groups)
    comp = {g: np.bincount(labels[groups == g], minlength=n_cls).astype(float) for g in gids}
    order = sorted(gids, key=lambda g: (-comp[g][3], -comp[g][2], -comp[g].sum(), g))
    split_of = {}
    for g in order:
        scores = [float((comp[g] * (cur[s] / np.maximum(target[s], 1.0))).sum()) for s in range(len(fractions))]
        s = int(np.argmin(scores)); split_of[g] = s; cur[s] += comp[g]
    # local search: move single groups between splits while the relative per-class deviation from the 70/15/15 targets shrinks
    def cost(c): return float((((c - target) / np.maximum(target, 1.0)) ** 2).sum())
    improved = True
    while improved:
        improved = False
        for g in order:
            s0 = split_of[g]; best = (cost(cur), s0)
            for s1 in range(len(fractions)):
                if s1 == s0: continue
                trial = cur.copy(); trial[s0] -= comp[g]; trial[s1] += comp[g]
                cst = cost(trial)
                if cst < best[0] - 1e-12: best = (cst, s1)
            if best[1] != s0:
                cur[s0] -= comp[g]; cur[best[1]] += comp[g]; split_of[g] = best[1]; improved = True
    return np.array([split_of[g] for g in groups])

def leakage_report(X, y, tr, te, name):
    """For every test image: correlation with its most similar TRAIN image, and 1-nearest-neighbour accuracy against train."""
    S = X[te] @ X[tr].T
    nn = S.argmax(axis=1); mx = S.max(axis=1)
    acc = float((y[tr][nn] == y[te]).mean())
    r = {"split": name, "p10_max_corr": float(np.percentile(mx, 10)), "median_max_corr": float(np.median(mx)), "p90_max_corr": float(np.percentile(mx, 90)),
         "frac_gt_0.99": float((mx > 0.99).mean()), "frac_gt_0.95": float((mx > 0.95).mean()), "frac_gt_0.90": float((mx > 0.90).mean()), "nn1_accuracy": acc}
    print(f"{name:36s} corr p10/med/p90 {r['p10_max_corr']:.3f}/{r['median_max_corr']:.3f}/{r['p90_max_corr']:.3f} | frac>0.99 {r['frac_gt_0.99']:.3f} >0.95 {r['frac_gt_0.95']:.3f} >0.90 {r['frac_gt_0.90']:.3f} | 1-NN acc {100*acc:.2f}%")
    return r


In [ ]:
# 3. Build the 5 folds (grouped or random) and the validation-fold rule
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
df = df.sort_values("filepath").reset_index(drop=True)
y_all = np.array([label_to_idx[l] for l in df["label"]])
if SPLIT_MODE == "grouped":
    X_feat = pixel_features(df["filepath"].tolist())
    if TAU_FIXED is None: group_ids, TAU, _ladder = threshold_groups(X_feat, max_frac=MAX_GROUP_FRAC)
    else: group_ids, TAU = groups_at_tau(X_feat, TAU_FIXED), TAU_FIXED
    fold_of = assign_groups(group_ids, y_all, fractions=(1.0 / N_FOLDS,) * N_FOLDS)
    print("Near-duplicate threshold tau =", TAU, "| groups:", len(np.unique(group_ids)), "| largest group:", int(np.bincount(group_ids).max()))
    mod_folds = sorted(set(fold_of[y_all == 3])); print("Folds containing ModerateDemented groups:", mod_folds)
    leak_tr = np.flatnonzero(fold_of != 0); leak_te = np.flatnonzero(fold_of == 0)
    leakage_report(X_feat, y_all, leak_tr, leak_te, "fold 0 test -> rest (grouped)")
else:
    group_ids = np.arange(len(df)); fold_of = np.zeros(len(df), dtype=int)
    for k, (_, te_ix) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SPLIT_SEED).split(df, y_all)): fold_of[te_ix] = k
for k in range(N_FOLDS):
    print(f"fold {k}: n={int((fold_of == k).sum())}  per-class", np.bincount(y_all[fold_of == k], minlength=4).tolist())

def val_fold_for(k):
    """first other fold (cyclic) whose removal still leaves >= 20 ModerateDemented images in training"""
    for j in [(k + i) % N_FOLDS for i in range(1, N_FOLDS)]:
        if int(((y_all == 3) & (fold_of != k) & (fold_of != j)).sum()) >= 20: return j
    return (k + 1) % N_FOLDS

def set_fold(k):
    """sets train_df / val_df / test_df / class_weight_dict as globals for fold k"""
    global train_df, val_df, test_df, class_weight_dict, CUR_VAL_FOLD
    vf = val_fold_for(k); CUR_VAL_FOLD = vf
    train_df = df[(fold_of != k) & (fold_of != vf)].sample(frac=1.0, random_state=SEED + k).reset_index(drop=True)   # shuffled: never class-ordered
    val_df = df[fold_of == vf].reset_index(drop=True)
    test_df = df[fold_of == k].reset_index(drop=True)
    labs = sorted(train_df["label"].unique())
    cw = compute_class_weight(class_weight="balanced", classes=np.array(labs), y=train_df["label"])
    class_weight_dict = {label_to_idx[a]: float(b) for a, b in zip(labs, cw)}
    return train_df, val_df, test_df

for k in range(N_FOLDS):
    tr_, va_, te_ = set_fold(k)
    print(f"fold {k}: test fold {k} | val fold {CUR_VAL_FOLD} | train/val/test {len(tr_)}/{len(va_)}/{len(te_)} | train per-class", np.bincount(labels_ := np.array([label_to_idx[l] for l in tr_["label"]]), minlength=4).tolist(), "| class weights", {a: round(b, 2) for a, b in class_weight_dict.items()})
set_fold(0)
print('Training-order check (must be mixed, not one class after another):', train_df['label'].head(10).tolist())


In [ ]:
# 4. Data pipeline. Augmentation lives INSIDE the model (Keras layers are active only during training),
# so validation/test pipelines are plain and deterministic.
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32)

def make_dataset(frame, training=False, seed=42, flip=False):
    paths = frame["filepath"].values
    labels = np.array([label_to_idx[l] for l in frame["label"].values])
    labels_oh = tf.keras.utils.to_categorical(labels, num_classes=len(CLASSES)).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_oh))
    ds = ds.map(lambda p, l: (load_image(p), l), num_parallel_calls=AUTOTUNE)
    if flip:
        ds = ds.map(lambda x, y: (tf.image.flip_left_right(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(max(len(frame), 2048), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ], name="augmentation")


In [ ]:
# 5. Model builders: one shared head design for every backbone, so the comparison is about the backbone only
from tensorflow.keras.applications import ResNet50, ConvNeXtTiny, EfficientNetV2S

def build_model(arch):
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = build_augmentation()(inputs)
    if arch == "resnet50":
        x = tf.keras.applications.resnet50.preprocess_input(x)
        base = ResNet50(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
    elif arch == "convnext_tiny":      # preprocessing is built into the Keras ConvNeXt (expects 0-255 input)
        base = ConvNeXtTiny(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
    elif arch == "efficientnetv2_s":   # preprocessing is built into Keras EfficientNetV2 (expects 0-255 input)
        base = EfficientNetV2S(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
    elif arch == "fusion":             # EfficientNetB0 + ResNet50 feature-level fusion (same idea as Experiments 1-2)
        from tensorflow.keras.applications import EfficientNetB0
        eff = EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=x, pooling=None)
        eff._name = "efficientnetb0_branch"
        xr = tf.keras.applications.resnet50.preprocess_input(x)
        res = ResNet50(include_top=False, weights=WEIGHTS, input_tensor=xr, pooling=None)
        res._name = "resnet50_branch"
        f_eff = layers.GlobalAveragePooling2D(name="eff_gap")(eff.output)
        f_res = layers.GlobalAveragePooling2D(name="res_gap")(res.output)
        feat = layers.Concatenate(name="feature_fusion")([f_eff, f_res])
        base = [eff, res]
    else:
        raise ValueError(arch)
    if arch != "fusion":
        feat = layers.GlobalAveragePooling2D(name="gap")(base.output)
    h = layers.Dropout(0.3)(feat)
    h = layers.Dense(256, activation="relu")(h)
    h = layers.Dropout(0.3)(h)
    outputs = layers.Dense(len(CLASSES), activation="softmax", dtype="float32", name="predictions")(h)
    model = models.Model(inputs, outputs, name=arch)
    for bb in (base if isinstance(base, list) else [base]):
        for layer in bb.layers:
            layer.trainable = False
    return model, base

def unfreeze_all_but_bn(base):
    for bb in (base if isinstance(base, list) else [base]):
        for layer in bb.layers:
            if not isinstance(layer, layers.BatchNormalization):
                layer.trainable = True

def make_loss():
    return tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

def make_phase2_optimizer(steps_total):
    # v2 fix: the first pilot collapsed at peak LR 1e-4 (model predicted a single class). 2e-5 + gradient clipping is stable
    # with the heavy ModerateDemented class weight (24.9) used since Experiments 1-2.
    schedule = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=2e-5, decay_steps=max(steps_total, 1), alpha=0.05)
    try:
        return optimizers.AdamW(learning_rate=schedule, weight_decay=1e-4, clipnorm=1.0)
    except Exception:
        return optimizers.Adam(learning_rate=schedule, clipnorm=1.0)

def train_two_phase(model, base, train_ds, val_ds):
    # Phase 1: head only
    model.compile(optimizer=optimizers.Adam(1e-3, clipnorm=1.0), loss=make_loss(), metrics=["accuracy"])
    cb1 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=0)]
    model.fit(train_ds, validation_data=val_ds, epochs=P1_EPOCHS, class_weight=class_weight_dict, callbacks=cb1, verbose=2)
    # Phase 2: fine-tune everything except BatchNorm, AdamW + cosine decay
    unfreeze_all_but_bn(base)
    steps_total = int(np.ceil(len(train_df) / BATCH_SIZE)) * P2_EPOCHS
    model.compile(optimizer=make_phase2_optimizer(steps_total), loss=make_loss(), metrics=["accuracy"])
    cb2 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=0),
           callbacks.TerminateOnNaN()]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=P2_EPOCHS, class_weight=class_weight_dict, callbacks=cb2, verbose=2)
    best_val = float(max(hist.history["val_accuracy"]))
    return model, best_val

def predict_probs(model, ds):
    probs = model.predict(ds, verbose=0)
    return np.asarray(probs, dtype="float64")

def labels_of(frame):
    return np.array([label_to_idx[l] for l in frame["label"].values])

def metrics_for(y_true, probs):
    y_pred = np.argmax(probs, axis=1)
    y_oh = tf.keras.utils.to_categorical(y_true, num_classes=len(CLASSES))
    try:
        auc = float(roc_auc_score(y_oh, probs, average="macro", multi_class="ovr"))
    except Exception:
        auc = float("nan")
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro": auc,
    }


In [ ]:
# 6. MAIN LOOP over folds: train (validation-only selection) -> predict the held-out fold once (plain + flip-TTA); finished folds are skipped on re-run
RESULTS_CSV = f"{TAG}_per_fold_results.csv"
rows = pd.read_csv(RESULTS_CSV).to_dict("records") if os.path.exists(RESULTS_CSV) else []
arch = ARCHS[0]
for k in FOLDS:
    tag = f"{arch}_fold{k}"
    if os.path.exists(f"probs_tta_{tag}.npy"):
        print(f"[skip] {tag} already done"); continue
    tr_, va_, te_ = set_fold(k)
    print(f"\n===== {TAG} | {tag} | train {len(tr_)} val {len(va_)} test {len(te_)} (val fold {CUR_VAL_FOLD}) =====")
    t0 = time.time()
    tf.keras.backend.clear_session(); gc.collect()
    random.seed(SEED + k); np.random.seed(SEED + k); tf.random.set_seed(SEED + k)
    train_ds = make_dataset(train_df, training=True, seed=SEED + k)
    val_ds = make_dataset(val_df); test_ds = make_dataset(test_df); test_ds_flip = make_dataset(test_df, flip=True)
    model, base_m = build_model(arch)
    model, best_val = train_two_phase(model, base_m, train_ds, val_ds)
    p_plain = predict_probs(model, test_ds); p_flip = predict_probs(model, test_ds_flip); p_tta = (p_plain + p_flip) / 2.0
    y_te = labels_of(test_df)
    m_plain, m_tta = metrics_for(y_te, p_plain), metrics_for(y_te, p_tta)
    np.save(f"probs_plain_{tag}.npy", p_plain); np.save(f"probs_tta_{tag}.npy", p_tta); np.save(f"y_{tag}.npy", y_te)
    test_df.assign(fold=k).to_csv(f"files_{tag}.csv", index=False)
    row = {"arch": arch, "fold": k, "val_fold": CUR_VAL_FOLD, "n_test": len(test_df), "val_best_accuracy": best_val,
           "test_accuracy_plain": m_plain["accuracy"], "test_accuracy_tta": m_tta["accuracy"],
           "f1_macro_plain": m_plain["f1_macro"], "f1_macro_tta": m_tta["f1_macro"], "minutes": round((time.time() - t0) / 60, 1)}
    rows.append(row); pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    print(f"  {tag}: val_best={best_val:.4f} | test plain={m_plain['accuracy']:.4f} | test TTA={m_tta['accuracy']:.4f} | {row['minutes']} min")
    del model, base_m, train_ds, val_ds, test_ds, test_ds_flip
    tf.keras.backend.clear_session(); gc.collect()


In [ ]:
# 7. Pooled out-of-fold results (every image tested once) and per-fold table
import glob
res = pd.DataFrame(rows)
print(res[["fold", "n_test", "val_best_accuracy", "test_accuracy_plain", "test_accuracy_tta", "f1_macro_tta", "minutes"]].round(4).to_string(index=False))
ks = sorted(res["fold"].unique())
P_plain = np.concatenate([np.load(f"probs_plain_{arch}_fold{k}.npy") for k in ks]); P_tta = np.concatenate([np.load(f"probs_tta_{arch}_fold{k}.npy") for k in ks])
Y = np.concatenate([np.load(f"y_{arch}_fold{k}.npy") for k in ks])
files = pd.concat([pd.read_csv(f"files_{arch}_fold{k}.csv") for k in ks], ignore_index=True)
np.save(f"{TAG}_oof_probs_tta.npy", P_tta); np.save(f"{TAG}_oof_probs_plain.npy", P_plain); np.save(f"{TAG}_oof_y.npy", Y); files.to_csv(f"{TAG}_oof_files.csv", index=False)
summ = {}
for nm, P in [("plain", P_plain), ("tta", P_tta)]:
    m = metrics_for(Y, P); summ[nm] = m
    fa = res[f"test_accuracy_{nm}"].values
    print(f"POOLED {nm}: acc {100*m['accuracy']:.2f} | macro-F1 {100*m['f1_macro']:.2f} | macro-AUC {m['auc_macro']:.4f} | per-fold acc mean {100*fa.mean():.2f} +/- {100*fa.std(ddof=1) if len(fa)>1 else 0:.2f}")
summ["per_fold_acc_tta"] = res["test_accuracy_tta"].tolist(); summ["per_fold_acc_plain"] = res["test_accuracy_plain"].tolist()
json.dump(summ, open(f"{TAG}_summary.json", "w"), indent=2)


In [ ]:
# 8. Per-class report and pooled confusion matrix (flip-TTA)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
pred = P_tta.argmax(1)
labels_present = [i for i in range(4) if (Y == i).any()]
print(classification_report(Y, pred, labels=labels_present, target_names=[CLASSES[i] for i in labels_present], digits=4, zero_division=0))
cm_ = confusion_matrix(Y, pred, labels=list(range(4)))
print(cm_)
fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(cm_, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(CLASSES, rotation=30, ha="right"); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{TAG}: pooled out-of-fold confusion matrix")
for i in range(4):
    for j in range(4): ax.text(j, i, int(cm_[i, j]), ha="center", va="center", color="white" if cm_[i, j] > cm_.max()/2 else "black")
plt.tight_layout(); plt.savefig(f"{TAG}_confusion_matrix.png", dpi=200); plt.close()
json.dump({"confusion_matrix": cm_.tolist(), "report": classification_report(Y, pred, labels=labels_present, target_names=[CLASSES[i] for i in labels_present], digits=4, zero_division=0, output_dict=True)}, open(f"{TAG}_per_class.json", "w"), indent=2)
if SPLIT_MODE == "grouped":
    pd.DataFrame({"filepath": df["filepath"], "label": df["label"], "group": group_ids, "fold": fold_of}).to_csv(f"{TAG}_folds.csv", index=False)
print("Saved.")


## What to download and send back
From the Output tab: `exp11b_resnet50_gcv_per_fold_results.csv`, `exp11b_resnet50_gcv_summary.json`, `exp11b_resnet50_gcv_per_class.json`, `exp11b_resnet50_gcv_confusion_matrix.png`, `exp11b_resnet50_gcv_oof_probs_tta.npy`, `exp11b_resnet50_gcv_oof_probs_plain.npy`, `exp11b_resnet50_gcv_oof_y.npy`, `exp11b_resnet50_gcv_oof_files.csv`
(and `exp11b_resnet50_gcv_folds.csv` for grouped mode), plus a copy of the log text. If the Output tab offers only a zip, download that.